# Multimodal RAG — Free Hugging Face Models, CPU Embeddings, GPU-Safe

Complete Google Colab pipeline for PDF-based multimodal RAG.

**Design goals**
- No paid OpenAI models or API keys.
- Unstructured extracts text, tables, formulas, and images.
- Qwen3-VL handles multimodal enrichment and final grounded answers.
- `all-MiniLM-L6-v2` embeddings run strictly on CPU.
- ChromaDB stores vectors locally on disk.
- Qwen3-VL uses 4-bit loading when available to reduce T4 memory pressure.
- Every LangChain `Document.page_content` is forced to be a string.
- Small tests run before expensive full-document processing.

In [ ]:
# Install the packages required by this notebook.
# Run once in a fresh Colab runtime.

%pip install -q -U "unstructured[all-docs]"
%pip install -q -U langchain langchain-community langchain-chroma
%pip install -q -U sentence-transformers transformers accelerate bitsandbytes
%pip install -q -U qwen-vl-utils
%pip install -q -U "Pillow==11.3.0"

### After installation

If Colab says packages were updated, use **Runtime → Restart session** before continuing.

The pinned Pillow version is included because the notebook previously hit the
`PIL._typing` / `_Ink` import error.

In [ ]:
# Check the Colab runtime before loading models.

import sys
import torch
from PIL import Image

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Pillow:", Image.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        f"GPU memory: "
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )
else:
    print("WARNING: No GPU detected. Multimodal inference will be slow.")

In [ ]:
# Import all libraries used by the RAG pipeline.

import os
import gc
import json
import shutil
import time
from typing import List, Optional

import torch

from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

from langchain_core.documents import Document
from langchain_chroma import Chroma

from sentence_transformers import SentenceTransformer

from transformers import (
    AutoProcessor,
    Qwen3VLForConditionalGeneration,
)

print("✅ Imports completed")

In [ ]:
# Central configuration.
# Change the PDF path here if your file has a different name.

PDF_PATH = "/content/Attention_is_all_you_need.pdf"

VLM_MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"

# Small CPU embedding model. It deliberately stays off the GPU.
EMBEDDING_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"

CHROMA_DIR = "/content/chroma_multimodal_rag"
TOP_K = 3

if not os.path.exists(PDF_PATH):
    raise FileNotFoundError(
        f"PDF not found: {PDF_PATH}. Upload the PDF to Colab first."
    )

print("PDF:", PDF_PATH)
print("VLM:", VLM_MODEL_ID)
print("Embeddings:", EMBEDDING_MODEL_ID)

In [ ]:
# Clear temporary CUDA memory before loading the multimodal model.

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("✅ Memory cleanup completed")

In [ ]:
# Extract text, tables, formulas, and images from the PDF.
# hi_res is used because the document contains structured visual content.

def partition_document(file_path: str):
    print(f"📄 Partitioning document: {file_path}")

    elements = partition_pdf(
        filename=file_path,
        strategy="hi_res",
        infer_table_structure=True,
        extract_image_block_types=["Image"],
        extract_image_block_to_payload=True,
    )

    print(f"✅ Extracted {len(elements)} elements")
    return elements


elements = partition_document(PDF_PATH)

In [ ]:
# Inspect the element types produced by Unstructured.

from collections import Counter

element_types = Counter(
    type(element).__name__
    for element in elements
)

print("Detected element types:")

for name, count in element_types.items():
    print(f"  {name}: {count}")

In [ ]:
# Count tables and images for pipeline metrics.

table_count = sum(
    type(element).__name__ == "Table"
    for element in elements
)

image_count = sum(
    type(element).__name__ == "Image"
    for element in elements
)

print("Tables detected:", table_count)
print("Images detected:", image_count)

In [ ]:
# Group related PDF elements into retrieval-friendly chunks.
# Moderate chunk sizes keep the final retrieved context manageable.

chunks = chunk_by_title(
    elements,
    max_characters=3000,
    new_after_n_chars=2400,
    combine_text_under_n_chars=500,
)

print(f"✅ Created {len(chunks)} chunks")

In [ ]:
# Safely read Unstructured chunks.
# ElementMetadata is an object, not a normal Python dictionary.

def get_chunk_text(chunk) -> str:
    return str(getattr(chunk, "text", "") or "")


def extract_multimodal_content(chunk):
    metadata = chunk.metadata

    text = get_chunk_text(chunk)

    tables = []
    table_html = getattr(metadata, "text_as_html", None)

    if table_html:
        tables.append(str(table_html))

    images = []
    image_base64 = getattr(metadata, "image_base64", None)

    if image_base64:
        images.append(str(image_base64))

    source = str(
        getattr(metadata, "filename", "unknown")
    )

    page_number = getattr(
        metadata,
        "page_number",
        None
    )

    return text, tables, images, source, page_number


print("✅ Chunk extraction helpers ready")

## Load Qwen3-VL

The earlier CUDA OOM occurred because Qwen3-VL was already using most of the T4
memory and the embedding model was then moved to CUDA.

This version keeps embeddings on CPU and loads Qwen3-VL in 4-bit when possible.

In [ ]:
# Load Qwen3-VL.
# 4-bit quantization reduces GPU memory usage on a Colab T4.

print("Loading Qwen3-VL...")

processor = AutoProcessor.from_pretrained(
    VLM_MODEL_ID
)

try:
    from transformers import BitsAndBytesConfig

    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    model = Qwen3VLForConditionalGeneration.from_pretrained(
        VLM_MODEL_ID,
        quantization_config=quantization_config,
        device_map="auto",
    )

    print("✅ Qwen3-VL loaded in 4-bit mode")

except Exception as error:
    print("⚠️ 4-bit loading failed:")
    print(error)
    print("Trying automatic dtype/device placement...")

    model = Qwen3VLForConditionalGeneration.from_pretrained(
        VLM_MODEL_ID,
        dtype="auto",
        device_map="auto",
    )

    print("✅ Qwen3-VL loaded with automatic placement")

model.eval()

if torch.cuda.is_available():
    print(
        f"Allocated GPU memory: "
        f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
    )

In [ ]:
# Generate text with Qwen3-VL.
# Images are optional; text-only generation uses the same function.

def generate_vlm_response(
    text: str,
    images: Optional[list] = None,
    max_new_tokens: int = 256,
):
    text = str(text or "")

    content = [
        {
            "type": "text",
            "text": text,
        }
    ]

    if images:
        for image in images:
            if image:
                content.append(
                    {
                        "type": "image",
                        "image": image,
                    }
                )

    messages = [
        {
            "role": "user",
            "content": content,
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    )

    target_device = next(model.parameters()).device

    inputs = {
        key: value.to(target_device)
        if hasattr(value, "to")
        else value
        for key, value in inputs.items()
    }

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
        )

    input_length = inputs["input_ids"].shape[1]
    generated_ids = outputs[:, input_length:]

    response = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
    )[0]

    response = str(response).strip()

    del inputs
    del outputs
    del generated_ids

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return response

In [ ]:
# Test Qwen before processing the complete PDF.

test_response = generate_vlm_response(
    "What is 2 + 2? Answer with only the number.",
    max_new_tokens=16,
)

print("Qwen response:", repr(test_response))

assert isinstance(test_response, str)
assert len(test_response) > 0

print("✅ Qwen3-VL test passed")

In [ ]:
# Create a searchable description for table/image chunks.
# The function always returns a string, preventing the previous
# Document(page_content=None) error.

def create_ai_enhanced_summary(
    text: str,
    tables: List[str],
    images: List[str],
):
    text = str(text or "")

    prompt = f"""
You are preparing content for a Retrieval-Augmented Generation system.

Create a concise, factual, searchable description of the supplied document chunk.

TEXT:
{text}
"""

    if tables:
        prompt += "\n\nTABLES:\n"

        for index, table in enumerate(tables, start=1):
            prompt += (
                f"\nTable {index}:\n"
                f"{table}\n"
            )

    prompt += """
Rules:
- Preserve important facts, numbers, names, and technical terms.
- Explain important table information clearly.
- Do not invent facts.
- Do not use outside knowledge.
- Return only the searchable description.
"""

    try:
        response = generate_vlm_response(
            prompt,
            images=images if images else None,
            max_new_tokens=256,
        )

        if response:
            return response

    except Exception as error:
        print(f"⚠️ Qwen enrichment failed: {error}")

    if text:
        return text

    if tables:
        return "\n".join(tables)

    return "Multimodal document content."

In [ ]:
# Test the summarizer on one table before processing all chunks.

table_test_chunk = None

for chunk in chunks:
    text, tables, images, source, page = (
        extract_multimodal_content(chunk)
    )

    if tables:
        table_test_chunk = chunk
        break

if table_test_chunk is None:
    print("ℹ️ No table chunk found; table test skipped.")
else:
    text, tables, images, source, page = (
        extract_multimodal_content(table_test_chunk)
    )

    summary = create_ai_enhanced_summary(
        text=text,
        tables=tables,
        images=[],
    )

    print("===== TABLE SUMMARY TEST =====")
    print(summary[:1500])

    assert isinstance(summary, str)
    assert len(summary) > 0

    print("✅ Table summarization test passed")

In [ ]:
# Process all chunks.
# Text-only chunks do not call the VLM unnecessarily.
# Table/image chunks are enriched for better retrieval.

def summarise_chunks(chunks):
    processed_chunks = []

    print(f"Processing {len(chunks)} chunks...")

    for index, chunk in enumerate(chunks, start=1):

        print(
            f"\nProcessing chunk {index}/{len(chunks)}"
        )

        text, tables, images, source, page = (
            extract_multimodal_content(chunk)
        )

        if tables or images:

            print(
                f"  → Multimodal chunk "
                f"(tables={len(tables)}, images={len(images)})"
            )

            summary = create_ai_enhanced_summary(
                text=text,
                tables=tables,
                images=images,
            )

        else:

            print("  → Text-only chunk")
            summary = text

        summary = str(summary or "").strip()

        if not summary:
            print("  ⚠️ Empty chunk skipped")
            continue

        metadata = {
            "source": source,
            "page_number": page,
            "has_table": bool(tables),
            "has_image": bool(images),
        }

        processed_chunks.append(
            Document(
                page_content=summary,
                metadata=metadata,
            )
        )

    print(
        f"\n✅ Processed "
        f"{len(processed_chunks)} chunks"
    )

    return processed_chunks

In [ ]:
# Run multimodal enrichment for the whole document.

processed_chunks = summarise_chunks(chunks)

assert len(processed_chunks) > 0

# Validate page_content before creating embeddings.
for document in processed_chunks:
    assert isinstance(document.page_content, str)
    assert document.page_content.strip()

print("✅ All processed chunks have valid string page_content")

## CPU embeddings and local ChromaDB

The embedding model is intentionally CPU-only. This directly addresses the
CUDA OOM seen when the embedding model tried to allocate GPU memory while
Qwen3-VL was already loaded.

ChromaDB itself stores the vectors locally and does not need GPU memory.

In [ ]:
# Load the small embedding model strictly on CPU.

print("Loading embedding model on CPU...")

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_ID,
    device="cpu",
)

print("✅ CPU embedding model loaded")

In [ ]:
# Adapt the CPU embedding model to the interface ChromaDB expects.

class CPUEmbeddingFunction:

    def __init__(self, model):
        self.model = model

    def embed_documents(self, texts):
        embeddings = self.model.encode(
            texts,
            batch_size=8,
            normalize_embeddings=True,
            show_progress_bar=True,
            convert_to_numpy=True,
        )

        return embeddings.tolist()

    def embed_query(self, text):
        embedding = self.model.encode(
            [str(text)],
            normalize_embeddings=True,
            convert_to_numpy=True,
        )[0]

        return embedding.tolist()

    def __call__(self, input):
        return self.embed_documents(input)


embedding_function = CPUEmbeddingFunction(
    embedding_model
)

print("✅ CPU embedding function ready")

In [ ]:
# Create a fresh local Chroma directory.
# This avoids conflicts with a database built using a different embedding model.

if os.path.exists(CHROMA_DIR):
    shutil.rmtree(CHROMA_DIR)

os.makedirs(CHROMA_DIR, exist_ok=True)

print("Chroma storage:", CHROMA_DIR)

In [ ]:
# Create ChromaDB using CPU embeddings.
# The embedding model never touches the GPU.

start_time = time.time()

print("Creating ChromaDB...")

db = Chroma.from_documents(
    documents=processed_chunks,
    embedding=embedding_function,
    persist_directory=CHROMA_DIR,
    collection_metadata={
        "hnsw:space": "cosine",
    },
)

elapsed = time.time() - start_time

print(f"✅ ChromaDB created in {elapsed:.2f} seconds")
print(f"Stored chunks: {len(processed_chunks)}")

In [ ]:
# Test semantic retrieval before generating answers.

retriever = db.as_retriever(
    search_kwargs={
        "k": TOP_K,
    }
)

test_query = "What is the Transformer architecture?"

retrieved = retriever.invoke(test_query)

print(f"Retrieved {len(retrieved)} chunks")

for index, document in enumerate(retrieved, start=1):
    print(f"\n===== Retrieved chunk {index} =====")
    print(document.page_content[:800])

assert len(retrieved) > 0

print("\n✅ Retrieval test passed")

In [ ]:
# Generate a grounded answer from retrieved chunks.
# Qwen3-VL is instructed to use only the retrieved context.

def generate_final_answer(
    query: str,
    retrieved_documents: List[Document],
):

    context_parts = []

    for index, document in enumerate(
        retrieved_documents,
        start=1,
    ):
        context_parts.append(
            f"""
DOCUMENT {index}
Source: {document.metadata.get("source", "unknown")}
Page: {document.metadata.get("page_number", "unknown")}

{document.page_content}
"""
        )

    context = "\n".join(context_parts)

    prompt = f"""
You are a grounded Retrieval-Augmented Generation assistant.

Answer the user's question using ONLY the retrieved document content.

USER QUESTION:
{query}

RETRIEVED DOCUMENTS:
{context}

Rules:
1. Do not use outside knowledge.
2. Do not invent facts.
3. If the answer is not supported by the retrieved documents, say:
"I don't have enough information to answer that question based on the provided documents."
4. Give a clear and concise answer.
"""

    return generate_vlm_response(
        prompt,
        max_new_tokens=384,
    )

In [ ]:
# End-to-end RAG query:
# question → Chroma retrieval → Qwen3-VL grounded answer.

def ask_question(query: str):

    print("\n" + "=" * 60)
    print("QUESTION")
    print("=" * 60)
    print(query)

    retrieved_documents = retriever.invoke(query)

    print(
        f"\n📚 Retrieved {len(retrieved_documents)} chunks"
    )

    answer = generate_final_answer(
        query,
        retrieved_documents,
    )

    print("\n" + "=" * 60)
    print("ANSWER")
    print("=" * 60)
    print(answer)

    return answer


answer = ask_question(
    "What are the main components of the Transformer architecture?"
)

In [ ]:
# Run a few questions to validate retrieval and generation.

test_questions = [
    "What is self-attention?",
    "Why are positional encodings used?",
    "What is the role of multi-head attention?",
]

for question in test_questions:
    ask_question(question)

## Pipeline metrics

These are engineering health checks, not a benchmark of model quality.

In [ ]:
# Collect simple pipeline health metrics.

metrics = {
    "pdf_exists": os.path.exists(PDF_PATH),
    "elements_extracted": len(elements),
    "chunks_created": len(chunks),
    "chunks_processed": len(processed_chunks),
    "tables_detected": table_count,
    "images_detected": image_count,
    "chromadb_exists": os.path.exists(CHROMA_DIR),
    "retrieved_for_test_query": len(retrieved),
    "embedding_model": EMBEDDING_MODEL_ID,
    "vlm_model": VLM_MODEL_ID,
    "embedding_device": "cpu",
    "chroma_storage": "local disk",
}

print("===== PIPELINE METRICS =====")

for key, value in metrics.items():
    print(f"{key}: {value}")

In [ ]:
# Save the metrics as a JSON report.

metrics_path = "/content/rag_metrics.json"

with open(metrics_path, "w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=2)

print(f"✅ Metrics saved to {metrics_path}")

## Optional GPU cleanup

Run this only when you are finished asking questions.

In [ ]:
# Release Qwen3-VL GPU memory after the notebook is finished.

del model
del processor

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("✅ Qwen3-VL GPU memory released")